# Figure 2 — the dense transition

Train a block-sparse network until it stabilizes, then flip its connection
masks to all-ones so the cross-task weights become trainable. Those weights are
already exactly zero, so the flip does not change the function the network
computes — only which weights gradients can reach.

This separates two explanations for the gap in Figure 1. If dense networks
merely fail to *find* the block-sparse solution, then handing one that solution
and continuing should be harmless. It is not: performance degrades shortly
after the new connections appear, and at its own best step-size it eventually
falls below the plain dense network. That is what points at credit assignment
rather than an unlucky optimization path.

16-task multi-MNIST, 256 hidden units, one task re-permuted every 125 steps,
30 seeds. Produced by `experiments/sweeps/02_dense_transition/`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from multi_mnist.analysis import (
    asymptotic_metric, fetch_per_seed_arrays, filter_sweep, filter_to_best,
    get_color_palette, label_curves_inline, load_export, measure_inline_labels,
    normalize_columns, plot_ci_curve, plot_sensitivity, save_fig, set_style,
    stack_per_seed,
)

%matplotlib inline


In [ ]:
# The Comet project the published runs live in. Change it if you
# re-run the sweeps into your own project.
PROJECT = 'paper-weight-pruning-scaling-sweep'

# Both this repository's sweep name and the names the published runs were
# logged under, so the figure regenerates from either.
SWEEP_NAMES = [
    '03_dense_transition_n16_h256',                     # this repo
    '00_block_sparse_dense_fill_main_n16_h256',         # published runs
    '00_block_sparse_dense_fill_main_n16_h256_lrext',   # its step-size addendum
]

CELL_COLS = ['task.n_tasks', 'model.initial_hidden_units',
             'algorithm.transition_step', 'seed_offset', 'optimizer.learning_rate']

# The transition step is the method axis. 600,000 exceeds total_steps, so the
# fill never fires and the run is the block-sparse baseline.
NEVER = 600_000
TRANSITION = 250_000
METHOD_LABELS = {0: 'Dense', TRANSITION: 'Dense Transition', NEVER: 'Block-Sparse'}
PALETTE = get_color_palette(classes=list(METHOD_LABELS.values()))
N_TASKS = 16


In [ ]:
# Run once to fetch from Comet, then leave commented out.
# from multi_mnist.analysis import download_project
# download_project(PROJECT, output_dir='data')

cfg_all, run_all = load_export(PROJECT, data_dir='data')
cfg_all = normalize_columns(cfg_all)

cfg, run = filter_sweep(cfg_all, run_all, SWEEP_NAMES, cell_cols=CELL_COLS)
run['loss_per_task'] = run['loss'] / N_TASKS
print(f'{len(cfg)} trials after dedupe')
print(cfg.groupby(['algorithm.transition_step', 'seed_offset']).size())


## Step-size sensitivity

Each method uses mean accuracy over the final 10% of steps. Dense and dense-transition runs diverge above 2^-10.


In [ ]:
asym = asymptotic_metric(cfg, run, 'accuracy', ['algorithm.transition_step'], tail_frac=0.10)
asym['Method'] = asym['algorithm.transition_step'].map(METHOD_LABELS)

set_style('2-col')
fig, ax = plt.subplots()
plot_sensitivity(
    asym, ax=ax, hue_col='Method',
    palette=[PALETTE[m] for m in sorted(asym['Method'].dropna().unique())],
    title='Step-Size Sensitivity', ylabel='Accuracy', ylim=(0, 1),
    legend_title=None)
fig.tight_layout()
save_fig(fig_dir='../figures/generated', name='fig3_dense_transition_sensitivity')
plt.show()


## Forward-only control

The forward-only control blocks backward gradients through the added output connections. The added connections still contribute to the forward pass and update their weights.

Compare it with the full transition at the same step-size to separate forward noise from backward interference. The experiment uses `sweep/forward_only.yaml`.


In [ ]:
from multi_mnist.analysis import dedupe_trials

FORWARD_ONLY_SWEEP = '03_dense_transition_forward_only_n16_h256'
FORWARD_ONLY = 'Forward-Only Transition'
FO_COL = 'algorithm.forward_only_distractors'


def add_method_column(cfg):
    """Label each trial by arm.

    The forward-only runs share their `transition_step` with the ordinary
    ones, so the flag has to join the key or the two collapse into one cell.
    Runs from before the flag existed have no column at all.
    """
    out = cfg.copy()
    flag = (out[FO_COL].astype(str).str.lower().isin(['true', '1', '1.0'])
            if FO_COL in out.columns else pd.Series(False, index=out.index))
    out['Method'] = pd.to_numeric(
        out['algorithm.transition_step'], errors='coerce').map(METHOD_LABELS)
    out.loc[flag, 'Method'] = FORWARD_ONLY
    return out


# CELL_COLS with Method standing in for the transition step.
CELL_COLS_FO = ['task.n_tasks', 'model.initial_hidden_units',
                'Method', 'seed_offset', 'optimizer.learning_rate']

# Every method keeps its pinned color. Slot 2 already labels the
# step-size-matched transition curve in Figure 3, so the new arm takes 4.
PALETTE_FO = dict(PALETTE)
PALETTE_FO[FORWARD_ONLY] = get_color_palette()[4]


In [ ]:
cfg_fo, run_fo = filter_sweep(
    cfg_all, run_all, SWEEP_NAMES + [FORWARD_ONLY_SWEEP], dedup=False)
cfg_fo = add_method_column(cfg_fo)
cfg_fo, run_fo = dedupe_trials(cfg_fo, run_fo, CELL_COLS_FO)
run_fo['loss_per_task'] = run_fo['loss'] / N_TASKS
print(f'{len(cfg_fo)} trials after dedupe')
print(cfg_fo.groupby(['Method', 'seed_offset']).size())


## Best step-size per method

The forward-only arm only ran at two step-sizes, so its "best" is a choice
between those two rather than a swept optimum.


In [ ]:
best_cfg_fo, best_run_fo, best_fo = filter_to_best(
    cfg_fo, run_fo, ['Method'], 'accuracy', direction='max', tail_frac=0.10)

summary_fo = best_fo.copy()
summary_fo['step-size'] = summary_fo['optimizer.learning_rate'].apply(
    lambda v: f'2^{int(np.log2(float(v)))}')
print(summary_fo[['Method', 'step-size', '_metric']].to_string(index=False))

assets_fo = fetch_per_seed_arrays(best_cfg_fo['run_id'].tolist(),
                                  cache_dir='data/per_seed_cache')


## Prepare the five curves

Use each method at its selected step-size, plus the transition at the block-sparse step-size.


In [ ]:
# Numeric columns arrive from the CSV as objects, so coerce before comparing.
lr_bs = float(best_fo.loc[best_fo['Method'] == METHOD_LABELS[NEVER],
                          'optimizer.learning_rate'].iloc[0])
cfg_lr_fo = pd.to_numeric(cfg_fo['optimizer.learning_rate'], errors='coerce')


def _accs(run_ids):
    _, accs = stack_per_seed(
        fetch_per_seed_arrays(run_ids, cache_dir='data/per_seed_cache'), run_ids)
    return accs


# Each arm at its own best step-size, plus the ordinary transition re-run at
# block-sparse's step-size -- the curve that isolates the flip from any
# step-size re-tuning. Forward-only's own best is the same cell the ordinary
# transition wins at, so those two are directly comparable.
PAPER_CURVES = []
for method in (METHOD_LABELS[0], METHOD_LABELS[TRANSITION], METHOD_LABELS[NEVER],
               FORWARD_ONLY):
    rids = best_cfg_fo[best_cfg_fo['Method'] == method]['run_id'].tolist()
    PAPER_CURVES.append((method, _accs(rids), PALETTE_FO[method]))

matched_rids = cfg_fo[(cfg_fo['Method'] == METHOD_LABELS[TRANSITION])
                      & (np.abs(cfg_lr_fo - lr_bs) < lr_bs * 1e-6)]['run_id'].tolist()
PAPER_CURVES.insert(3, (
    f'Dense Transition ($2^{{{int(np.log2(lr_bs))}}}$)',
    _accs(matched_rids), get_color_palette()[2]))



## Figure 2 learning curves

Export the final panel with inline labels and the forward-only control.


In [ ]:
FIG_DIR = '../figures/generated'

# (text, x, side, optional smaller parenthetical) for each arm. The label
# positions are the only hand-set numbers; `label_curves_inline` works out the
# heights so every label ends up the same clearance from its own curve.
CURVE_LABELS = {
    METHOD_LABELS[0]:          ('Dense', 0.15e5, 'below', None),
    METHOD_LABELS[TRANSITION]: ('Dense transition', 0.16e5, 'above', None),
    METHOD_LABELS[NEVER]:      ('Block-sparse', 3.7e5, 'above', None),
    PAPER_CURVES[3][0]:        ('Dense transition', 0.17e5, 'above',
                                f'(step-size = $2^{{{int(np.log2(lr_bs))}}}$)'),
    FORWARD_ONLY:              ('Dense-transition (FO)', 3.05e5, 'above', None),
}


def plot_fig3_curves(savename, forward_only):
    """Figure 3's left panel, labelled inline. `forward_only` adds the fifth arm."""
    curves = [c for c in PAPER_CURVES if forward_only or c[0] != FORWARD_ONLY]
    # Forward-only is drawn first so the ordinary transition sits on top of it:
    # the two are the same run until the flip, and the panel is about where they
    # part, which the arm underneath would otherwise hide.
    curves.sort(key=lambda c: c[0] != FORWARD_ONLY)

    set_style('2-col')
    fig, ax = plt.subplots()
    for _, accs, color in curves:
        plot_ci_curve(np.arange(accs.shape[0]) * 1000, accs, color=color, ax=ax)
    ax.axvline(TRANSITION, linestyle=':', color='black', alpha=0.6, linewidth=1)
    ax.set_xlabel('Step'); ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1.0)
    # The x axis runs exactly as far as the runs do, as in the other panels.
    ax.set_xlim(0, max(len(a) for _, a, _ in curves) * 1000)
    ax.set_title('Learning Curves')
    ax.grid(True); ax.set_axisbelow(True)
    ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0), useMathText=True)
    fig.tight_layout()

    specs = []
    for name, accs, color in curves:
        text, x, side, sub = CURVE_LABELS[name]
        mean = accs.mean(axis=1)
        keep = ~np.isnan(mean)          # the 2^-8 arm diverges and its curve stops
        curve = (np.arange(len(mean))[keep] * 1000, mean[keep])
        specs.append((text, x, side, color, curve, sub) if sub
                     else (text, x, side, color, curve))
    texts = label_curves_inline(ax, specs)
    print(savename, measure_inline_labels(ax, specs, texts))

    # No legend to strip, so this writes one file per format rather than
    # save_fig's pair.
    for fmt in ('pdf', 'png'):
        fig.savefig(f'{FIG_DIR}/{fmt}/{savename}.{fmt}', bbox_inches='tight', dpi=400)
    plt.show()


plot_fig3_curves('paper_fig3_curves_fo_labeled', forward_only=True)